# Notebook 07: LeNet-5

**Module:** 06 CNN  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Understand LeNet-5 architecture (1998)
2. Key innovation: First successful CNN for digit recognition
3. Implement in PyTorch
4. Train/evaluate on CIFAR-10 subset

## LeNet-5 (1998)

**Key innovation:** First successful CNN for digit recognition

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
classes = train_set.classes
print(f'CIFAR-10: {len(train_set)} train, {len(test_set)} test, {len(classes)} classes')

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total, correct, loss_sum = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        total += len(y)
    return loss_sum/total, correct/total

def evaluate(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            loss = criterion(out, y)
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum/total, correct/total

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(16*6*6, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(), nn.Linear(84, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

model = LeNet5()
print(model)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

## Training (3 epochs demo on subset — run full training for benchmark)

In [ ]:
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Subset for demo speed
from torch.utils.data import Subset
subset = Subset(train_set, range(2000))
sub_loader = DataLoader(subset, batch_size=64, shuffle=True)

for epoch in range(3):
    tr_loss, tr_acc = train_epoch(model, sub_loader, optimizer, criterion)
    te_loss, te_acc = evaluate(model, test_loader, criterion)
    print(f'Epoch {epoch+1}: train_acc={tr_acc:.3f}, test_acc={te_acc:.3f}')

---

## Summary

LeNet-5: First successful CNN for digit recognition

**Next:** [08_AlexNet.ipynb](08_AlexNet.ipynb)